# Mini Project 01 — Synthetic Variant Calling Pipeline

**Module:** 09 — Genomics and NGS  
**Notebook:** 14 of 16  
**Portfolio artifact:** `portfolio/module09_variant_calling_pipeline.png`  
**Time estimate:** 3 hours

> **Pass-3 target:** Implement the full pipeline — reference simulation, read
> generation, pileup, genotype likelihoods, VCF output — from memory, without
> referring to NB07 or any other notebook in this module. Write it fresh.

---
## Project Overview

Build an end-to-end synthetic variant calling pipeline:

1. Simulate a 500 bp reference sequence
2. Plant 5 known SNPs at defined positions
3. Simulate 100× coverage Illumina-like reads with 0.5% error rate
4. Build a pileup from scratch
5. Call SNPs using genotype likelihoods (from scratch, no reference to NB07)
6. Write a valid VCF4.2 output file
7. Evaluate: compute precision and recall against planted SNPs
8. Produce a 4-panel portfolio figure

**Rules:**
- No pysam — SAM parsing from scratch
- Genotype likelihood model implemented from scratch
- VCF output must follow VCF4.2 spec
- All 5 planted SNPs must be recovered at FDR-controlled call set

In [ ]:
# STEP 1 & 2: Simulate reference with planted SNPs
import numpy as np
from collections import defaultdict
from dataclasses import dataclass
from itertools import combinations_with_replacement
import re

rng = np.random.default_rng(42)

REF_LEN = 500
BASES = ['A', 'C', 'G', 'T']

# Generate reference
ref_seq = ''.join(rng.choice(BASES, REF_LEN))

# Plant 5 known heterozygous SNPs
PLANTED_SNPS = {
    50:  ('planted_ref', 'planted_alt'),  # will be filled in below
    120: ('planted_ref', 'planted_alt'),
    250: ('planted_ref', 'planted_alt'),
    350: ('planted_ref', 'planted_alt'),
    450: ('planted_ref', 'planted_alt'),
}

planted_snps = {}
for pos in sorted(PLANTED_SNPS.keys()):
    ref_base = ref_seq[pos]
    alt_base = rng.choice([b for b in BASES if b != ref_base])
    planted_snps[pos] = (ref_base, alt_base)

print("Reference length:", REF_LEN)
print("Planted SNPs:")
for pos, (ref_b, alt_b) in sorted(planted_snps.items()):
    print(f"  pos={pos}: {ref_b}→{alt_b}")

In [ ]:
# STEP 3: Simulate reads
TARGET_COVERAGE = 100
READ_LEN = 75
ERROR_RATE = 0.005

n_reads = int(TARGET_COVERAGE * REF_LEN / READ_LEN)
print(f"Simulating {n_reads} reads for {TARGET_COVERAGE}× coverage")


@dataclass
class Read:
    read_id: str
    pos: int      # 1-based SAM position
    seq: str
    qual: str     # Phred+33
    strand: str
    cigar: str


def simulate_reads(ref, planted_snps, n_reads, read_len, error_rate, seed=42):
    rng = np.random.default_rng(seed)
    reads = []
    for i in range(n_reads):
        start = int(rng.integers(0, max(1, len(ref) - read_len + 1)))
        end = min(start + read_len, len(ref))
        bases = list(ref[start:end])
        actual_len = end - start

        # Apply planted SNPs (heterozygous: 50% chance for each read)
        for pos, (ref_b, alt_b) in planted_snps.items():
            if start <= pos < end:
                if rng.random() < 0.5:
                    bases[pos - start] = alt_b

        # Add sequencing errors
        for j in range(actual_len):
            if rng.random() < error_rate:
                bases[j] = rng.choice([b for b in BASES if b != bases[j]])

        # Quality scores
        quals = [chr(int(rng.integers(25, 38)) + 33) for _ in range(actual_len)]
        strand = '+' if rng.random() < 0.5 else '-'

        reads.append(Read(
            read_id=f'read_{i:06d}',
            pos=start + 1,
            seq=''.join(bases),
            qual=''.join(quals),
            strand=strand,
            cigar=f'{actual_len}M',
        ))
    return reads


reads = simulate_reads(ref_seq, planted_snps, n_reads, READ_LEN, ERROR_RATE)
print(f"Generated {len(reads)} reads")
print(f"Sample read: pos={reads[0].pos} cigar={reads[0].cigar} seq={reads[0].seq[:20]}...")

In [ ]:
# STEP 4: Build pileup

@dataclass
class PileupPos:
    pos: int
    ref: str
    bases: list
    quals: list
    strands: list

    @property
    def depth(self): return len(self.bases)

    @property
    def counts(self): return defaultdict(int, {b: self.bases.count(b) for b in BASES})

    @property
    def vaf(self):
        ref_count = self.bases.count(self.ref)
        return 1 - ref_count / self.depth if self.depth > 0 else 0


def build_pileup(reads, ref_seq, min_mapq=0):
    pileup = {}
    for r in reads:
        start = r.pos - 1
        ops = re.findall(r'(\d+)([MIDNSHP=X])', r.cigar)
        q_pos = 0
        ref_pos = start
        for length_s, op in ops:
            n = int(length_s)
            if op == 'M':
                for k in range(n):
                    gpos = ref_pos + k
                    if 0 <= gpos < len(ref_seq) and q_pos + k < len(r.seq):
                        if gpos not in pileup:
                            pileup[gpos] = PileupPos(gpos, ref_seq[gpos], [], [], [])
                        q = ord(r.qual[q_pos + k]) - 33 if q_pos + k < len(r.qual) else 20
                        pileup[gpos].bases.append(r.seq[q_pos + k])
                        pileup[gpos].quals.append(q)
                        pileup[gpos].strands.append(r.strand)
                q_pos += n; ref_pos += n
    return pileup


pileup = build_pileup(reads, ref_seq)
print(f"Pileup positions: {len(pileup)}")
for pos, (ref_b, alt_b) in sorted(planted_snps.items())[:2]:
    p = pileup.get(pos)
    if p:
        print(f"  pos={pos}: depth={p.depth}, counts={dict(p.counts)}, VAF={p.vaf:.2f}")

In [ ]:
# STEP 5: Genotype likelihood-based SNP caller (from scratch)

def phred_to_prob(q):
    return 10 ** (-q / 10)


def genotype_log_likelihood(bases, quals, genotype):
    a1, a2 = genotype
    log_lik = 0.0
    for b, q in zip(bases, quals):
        eps = phred_to_prob(q)
        p = 0.5 * ((1 - eps) if b == a1 else eps / 3) + \
            0.5 * ((1 - eps) if b == a2 else eps / 3)
        log_lik += np.log10(p + 1e-300)
    return log_lik


def call_snp(pileup_pos, min_depth=8, min_vaf=0.1, min_gq=20):
    p = pileup_pos
    if p.depth < min_depth or p.vaf < min_vaf:
        return None
    genotypes = list(combinations_with_replacement(BASES, 2))
    lls = {g: genotype_log_likelihood(p.bases, p.quals, g) for g in genotypes}
    best_g = max(lls, key=lls.get)
    if best_g == (p.ref, p.ref):
        return None
    sorted_ll = sorted(lls.values(), reverse=True)
    gq = int(min(-10 * (sorted_ll[1] - sorted_ll[0]), 99)) if len(sorted_ll) > 1 else 99
    if gq < min_gq:
        return None
    alt_alleles = [a for a in set(best_g) if a != p.ref]
    if not alt_alleles:
        return None
    return {
        'pos': p.pos + 1,  # 1-based VCF
        'ref': p.ref,
        'alt': alt_alleles[0],
        'depth': p.depth,
        'vaf': p.vaf,
        'gt': f'{best_g[0]}/{best_g[1]}',
        'gq': gq,
        'ref_count': p.bases.count(p.ref),
        'alt_count': p.bases.count(alt_alleles[0]),
        'fwd_alt': sum(1 for b, s in zip(p.bases, p.strands) if b == alt_alleles[0] and s == '+'),
        'rev_alt': sum(1 for b, s in zip(p.bases, p.strands) if b == alt_alleles[0] and s == '-'),
    }


calls = []
for pos in sorted(pileup.keys()):
    result = call_snp(pileup[pos])
    if result:
        calls.append(result)

# Evaluate
planted_positions = set(pos + 1 for pos in planted_snps.keys())  # 1-based
called_positions = set(c['pos'] for c in calls)
tp = called_positions & planted_positions
fp = called_positions - planted_positions
fn = planted_positions - called_positions

precision = len(tp) / len(called_positions) if called_positions else 0
recall = len(tp) / len(planted_positions) if planted_positions else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Calls made: {len(calls)}")
print(f"True positives: {len(tp)} ({sorted(tp)})")
print(f"False positives: {len(fp)} ({sorted(fp)})")
print(f"False negatives: {len(fn)} ({sorted(fn)})")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")

In [ ]:
# STEP 6: Write VCF output

def write_vcf(calls, ref_len, sample_name='SIMULATED_SAMPLE'):
    lines = [
        '##fileformat=VCFv4.2',
        '##source=module09_synthetic_pipeline',
        f'##reference=simulated_{ref_len}bp',
        '##FILTER=<ID=PASS,Description="All filters passed">',
        '##INFO=<ID=DP,Number=1,Type=Integer,Description="Depth">',
        '##INFO=<ID=VAF,Number=1,Type=Float,Description="Variant allele frequency">',
        '##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">',
        '##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allele depth ref,alt">',
        '##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype quality">',
        f'#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\t{sample_name}',
    ]
    for c in sorted(calls, key=lambda x: x['pos']):
        qual = round(-10 * np.log10(max(1e-10, 10 ** (-(c['gq'] / 10)))), 1)
        info = f"DP={c['depth']};VAF={c['vaf']:.3f}"
        fmt = 'GT:AD:GQ'
        sample = f"{c['gt']}:{c['ref_count']},{c['alt_count']}:{c['gq']}"
        lines.append(f"chr1\t{c['pos']}\t.\t{c['ref']}\t{c['alt']}\t{c['gq']}\tPASS\t{info}\t{fmt}\t{sample}")
    return '\n'.join(lines)


vcf_str = write_vcf(calls, REF_LEN)
print("VCF output (first 15 lines):")
for line in vcf_str.split('\n')[:15]:
    print(' ', line)

In [ ]:
# STEP 7-8: Portfolio figure
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Coverage depth
ax = axes[0, 0]
cov = np.zeros(REF_LEN, dtype=int)
for r in reads:
    s = r.pos - 1
    e = min(s + len(r.seq), REF_LEN)
    cov[s:e] += 1
ax.fill_between(range(REF_LEN), cov, color='steelblue', alpha=0.7)
ax.axhline(TARGET_COVERAGE, color='red', ls='--', lw=1, label=f'{TARGET_COVERAGE}× target')
for pos in planted_snps:
    ax.axvline(pos, color='orange', lw=1.5, alpha=0.8)
ax.set_xlabel('Position (bp)')
ax.set_ylabel('Coverage depth')
ax.set_title(f'A. Coverage depth (mean={cov.mean():.1f}×)\n(orange lines = planted SNP positions)')
ax.legend(fontsize=8)

# Panel B: Pileup allele frequencies at SNP positions
ax2 = axes[0, 1]
snp_positions = sorted(planted_snps.keys())
for idx, pos in enumerate(snp_positions):
    p = pileup.get(pos)
    if p is None: continue
    ref_b, alt_b = planted_snps[pos]
    counts_dict = dict(p.counts)
    x_positions = [idx - 0.2, idx + 0.2]
    ref_c = counts_dict.get(ref_b, 0)
    alt_c = counts_dict.get(alt_b, 0)
    total = ref_c + alt_c + 1e-9
    ax2.bar(idx - 0.2, ref_c / total * 100, width=0.35, color='steelblue', label='REF' if idx == 0 else '')
    ax2.bar(idx + 0.2, alt_c / total * 100, width=0.35, color='tomato', label='ALT' if idx == 0 else '')
ax2.axhline(50, color='gray', ls='--', lw=1, label='50% expected (het)')
ax2.set_xticks(range(len(snp_positions)))
ax2.set_xticklabels([f'pos {p}' for p in snp_positions], fontsize=8)
ax2.set_ylabel('Allele frequency (%)')
ax2.set_title('B. Allele frequencies at planted SNP positions')
ax2.legend(fontsize=8)

# Panel C: Called vs. planted SNPs (precision/recall)
ax3 = axes[1, 0]
all_positions_in_calls = sorted({c['pos'] - 1 for c in calls})
category = []
vaf_vals = []
gq_vals = []
for c in sorted(calls, key=lambda x: x['pos']):
    g_pos = c['pos'] - 1
    is_tp = g_pos in planted_snps
    category.append('TP' if is_tp else 'FP')
    vaf_vals.append(c['vaf'])
    gq_vals.append(c['gq'])
colors = ['green' if c == 'TP' else 'red' for c in category]
ax3.scatter(vaf_vals, gq_vals, c=colors, s=80, alpha=0.8, edgecolors='black', lw=0.5)
ax3.set_xlabel('Variant allele frequency (VAF)')
ax3.set_ylabel('Genotype quality (GQ)')
ax3.set_title(f'C. Calls: precision={precision:.2f}, recall={recall:.2f}, F1={f1:.2f}\n'
              f'(Green=TP, Red=FP; {len(calls)} total calls)')
tp_patch = mpatches.Patch(color='green', label=f'True positives ({len(tp)})')
fp_patch = mpatches.Patch(color='red', label=f'False positives ({len(fp)})')
ax3.legend(handles=[tp_patch, fp_patch], fontsize=8)

# Panel D: Genotype likelihood for one SNP site
ax4 = axes[1, 1]
example_pos = list(planted_snps.keys())[0]
p_example = pileup.get(example_pos)
if p_example:
    genotypes = list(combinations_with_replacement(BASES, 2))
    lls = {g: genotype_log_likelihood(p_example.bases, p_example.quals, g) for g in genotypes}
    best_ll = max(lls.values())
    pl_vals = [-10 * (v - best_ll) for v in lls.values()]
    gt_labels = [f'{g[0]}/{g[1]}' for g in genotypes]
    bar_colors = ['gold' if pl == 0 else 'steelblue' for pl in pl_vals]
    ax4.bar(gt_labels, [min(v, 200) for v in pl_vals], color=bar_colors)
    ax4.set_xlabel('Genotype')
    ax4.set_ylabel('PL (Phred, 0=best)')
    ref_b, alt_b = planted_snps[example_pos]
    ax4.set_title(f'D. Genotype likelihoods at pos {example_pos+1}\n'
                  f'(planted {ref_b}→{alt_b} het; best genotype highlighted in gold)')
    ax4.tick_params(axis='x', rotation=70, labelsize=7)

plt.suptitle('Module 09 Mini-Project: Synthetic Variant Calling Pipeline\n'
             f'({REF_LEN} bp reference, {n_reads} reads, 5 planted SNPs)',
             fontsize=12, fontweight='bold')
plt.tight_layout()

os.makedirs('../../../../portfolio', exist_ok=True)
plt.savefig('../../../../portfolio/module09_variant_calling_pipeline.png',
            dpi=150, bbox_inches='tight')
plt.savefig('variant_calling_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Portfolio figure saved to portfolio/module09_variant_calling_pipeline.png")

---
## Self-assessment

After completing the pipeline without looking at NB07:

- [ ] Could derive the genotype likelihood formula from first principles
- [ ] Could write `pileup_snp_caller()` from memory
- [ ] VCF output passes `bcftools stats` without errors
- [ ] All 5 planted SNPs recovered (recall = 1.0)
- [ ] Precision > 0.8 (fewer than 1 FP per 4 TPs)
- [ ] Portfolio figure saved and interpretable to a non-expert

If any step required looking back at NB07, note it here and re-attempt that specific
function from memory before moving on.

---
*Next: `15_mini_project_rnaseq_de_analysis.ipynb`*